# Модуль 1 · Лабораторный практикум
## От обнаружения взаимодействия факторов до решения о новом эксперименте

**Формат:** три связанных исследовательских кейса, решения пишете вы.  
**Время:** ориентировочно 60–90 минут.  
**Обязательно:** сначала прочитайте разбор 01_guided_case.ipynb.  
**Результат:** собственный протокол анализа, график и проект следующего эксперимента.

Вы помогаете группе исследователей материалов сравнить варианты технологического процесса. Инженер просит выбрать условия с большей прочностью; коллега прислал смешанные единицы измерения; бюджет ограничивает дополнительную серию испытаний.

**Данные полностью синтетические**, разработаны только для освоения вычислительного эксперимента. Они не описывают реальную технологию, сертификат качества или рекомендацию к производству.

Ваша задача — не получить красивое число, а последовательно объяснить: какие данные использовать, что удалось выяснить, какой вывод преждевременен и что измерять дальше.

### Проверка окружения и правила работы

Запускайте ячейки сверху вниз в чистом ядре. Разрешено обращаться к первому блокноту как к справочнику, но решения переносите на новые данные.

Пометки TODO — ваше место для реализации. Пустая заготовка возвращает None и даёт сообщение «ещё не решено», а не ломает весь блокнот. Проверки в этом файле проверяют **формат и инварианты**, но не подсказывают скрытый численный ответ.

Вы можете использовать стандартный Python; код приложения здесь не разрабатываем — для этого в дальнейшем будет IDE.

In [ ]:
import math
import csv
import json
from pathlib import Path
import matplotlib.pyplot as plt

def checkpoint(label, value):
    if value is None:
        print(f"{label}: ещё не решено. Вернитесь к TODO.")
        return False
    print(f"{label}: заполнено, проверьте научный смысл результата.")
    return True

print("Рабочая папка:", Path.cwd())

## Кейс A. Можно ли выбрать новый режим термообработки?

Вторая учебная серия для другого условного покрытия (не смешивайте её с данными из первого блокнота):

- Фактор A: температура, 55 °C (код −1) или 75 °C (код +1).
- Фактор B: массовая доля добавки, 0,5% (−1) или 2,5% (+1).
- Измеренный в синтетическом примере отклик: прочность, МПа.

**Новая ситуация.** Инженер предполагает, что температура всегда даёт одинаковую прибавку прочности. Проверьте предположение на новых данных. Сформулируйте прогноз до запуска.

**Задание А1.** Реализуйте функцию расчёта среднего эффекта A, B и факторного эффекта AB. Сравните условный эффект температуры на низком и высоком уровне добавки. Результат представьте в МПа.

Не путайте факторный эффект AB и разность двух условных эффектов: у них различается коэффициент.

In [ ]:
case_A=[
    {"run":"A1","A":-1,"B":-1,"temp_C":55,"additive_pct":0.5,"strength_MPa":46.0},
    {"run":"A2","A":+1,"B":-1,"temp_C":75,"additive_pct":0.5,"strength_MPa":50.0},
    {"run":"A3","A":-1,"B":+1,"temp_C":55,"additive_pct":2.5,"strength_MPa":42.0},
    {"run":"A4","A":+1,"B":+1,"temp_C":75,"additive_pct":2.5,"strength_MPa":74.0},
]
assert {(r["A"],r["B"]) for r in case_A}=={(-1,-1),(1,-1),(-1,1),(1,1)}
print("Получено записей:",len(case_A))
for row in case_A:
    print(row)

In [ ]:
def estimate_effects(records, response_key="strength_MPa"):
    # TODO А1: верните словарь {"A": ..., "B": ..., "AB": ...}.
    # Подсказка: сравните средние отклики при A=+1 и A=-1,
    # затем при B=+1 и B=-1, наконец по знаку произведения A*B.
    return None

effects=estimate_effects(case_A)
if checkpoint("А1: эффекты",effects):
    assert isinstance(effects,dict) and set(("A","B","AB")).issubset(effects)
    assert all(isinstance(effects[k],(float,int)) and math.isfinite(effects[k]) for k in ("A","B","AB"))
    print("Ваши эффекты (МПа):",effects)

**Задание А2.** Постройте график «прочность — температура» с отдельными линиями для двух долей добавки. Подпишите оси и единицы. Поясните, что означает непараллельность линий. Почему график сам по себе не доказывает статистическую значимость?

**Задание А3.** Сформулируйте вывод в 3–5 предложениях: какое сочетание имеет самый высокий *наблюдаемый* отклик? Можно ли считать его устойчиво оптимальным? Какие наблюдения нужны для подтверждения?

In [ ]:
# TODO А2: создайте график с двумя линиями и легендой.
# Пример заготовки:
# fig, ax = plt.subplots(figsize=(7,4))
# ...
# ax.set_xlabel("Температура, °C")
# ax.set_ylabel("Прочность, МПа")
# plt.show()
print("А2: постройте график выше и объясните его в Markdown-ячейке ниже.")

**Ваш аналитический вывод по кейсу A** (заполнить после расчёта):  

Наблюдение: ...  

Есть/нет свидетельство взаимодействия в данных: ...  

Что нельзя утверждать без дополнительных измерений: ...  

Предлагаемое следующее действие: ...

## Кейс B. Ошибка в отчёте или в единицах?

Из другой лабораторной выгрузки пришли значения прочности в МПа и кПа. Если просто сравнить числа 46 и 50 000, алгоритм сделает абсурдный вывод.

**Задание B1.** Напишите преобразователь в МПа. Запретите неизвестные единицы; не используйте молчаливую подстановку. Приведите значения к одним единицам, сохранив исходный формат, и проверьте, согласуются ли данные с кейсом A.

**Скрытая деталь:** одинаковые числа после перевода единиц ещё не доказывают, что это независимые повторные измерения: источник и идентификаторы опыта должны быть проверены.

In [ ]:
mixed=[
    {"run":"A1","value":46.0,"unit":"MPa"},
    {"run":"A2","value":50000.0,"unit":"kPa"},
    {"run":"A3","value":42.0,"unit":"MPa"},
    {"run":"A4","value":74000.0,"unit":"kPa"},
]
def to_mpa(value,unit):
    # TODO B1: возвратите число в MPa, при неизвестной единице
    # бросьте ValueError("unknown unit"). Поддержите MPa и kPa.
    return None

converted=[]
for row in mixed:
    value=to_mpa(row["value"],row["unit"])
    if value is not None:
        assert math.isfinite(value) and value>0
    converted.append({**row,"strength_MPa":value})
checkpoint("B1: единицы приведены",None if any(r["strength_MPa"] is None for r in converted) else converted)
for row in converted:
    print(row)

**Задание B2.** Сравните результаты двух случаев: (a) ошибка масштаба из-за единиц; (b) повторное использование той же самой серии наблюдений под другой подписью. В чём различаются проблемы качества данных и как их обнаружить?

**Задание B3.** Предложите два поля метаданных, без которых нельзя считать выгрузку независимым подтверждением (например, идентификатор образца, дата, партия или измерительная методика). Обоснуйте выбор.

**Ваш протокол по кейсу B:**  

Какой вывод выглядел бы неверным при смешении единиц: ...  

Как доказать, что набор является независимым: ...  

Что в действительности можно проверить сейчас: ...

## Кейс C. Бюджет на дополнительные измерения

Доступно не более **шести новых измерений**. На основе одного значения в каждом сочетании факторов нельзя оценить дисперсию ошибки, проверить стабильность оптимума и статистически обосновать эффект взаимодействия.

**Задание C1.** Предложите конкретные дополнительные сочетания A/B и число повторов (суммарно не более 6). Обоснуйте, что позволит оценить ваш план и какие вопросы всё равно останутся открыты.

**Задание C2.** Рассмотрите альтернативный план, в котором часть бюджета отдана центральной точке (65 °C; 1,5% добавки). Что она сможет показать относительно линейной модели? Почему одной центральной точки недостаточно для уверенных статистических выводов?

**Задание C3.** Представьте, что новые измерения сильно различаются. Как это изменит ваше первоначальное решение о режиме производства?

*Замечание:* центральная точка и уровни факторов — учебные. Не предполагайте, что реальный физический процесс линейный и что дополнительные образцы независимы без проверки.

In [ ]:
# TODO C1: задайте план из 1–6 запусков.
# Формат каждой записи: {"A": -1 or 0 or 1, "B": -1 or 0 or 1,
#                        "purpose": "почему эта точка нужна"}
new_experiments = []
if not new_experiments:
    print("C1: спланируйте новые измерения. Никаких запусков пока не выполнено.")
else:
    assert len(new_experiments)<=6
    assert all(r["A"] in (-1,0,1) and r["B"] in (-1,0,1) and r.get("purpose","").strip() for r in new_experiments)
    print("План принят структурно; научная обоснованность требует вашей защиты.")
    for r in new_experiments: print(r)

## Итог: исследовательское решение и карта дисциплин

Подготовьте один короткий отчёт, отвечая на вопросы:

1. Что вы пытались выяснить? Какие величины контролировали, какие измеряли?
2. Какие данные пришлось исправить и почему исходные значения нельзя было напрямую сравнивать?
3. Где наблюдалось взаимодействие факторов и каким графиком вы это показали?
4. Чего вы **не** смогли установить по четырём точкам без повторов?
5. Какие шесть или меньше новых измерений вы назначили бы, чтобы уменьшить неопределённость?

**Карта знаний.** Добавьте обоснованные отношения: экспериментальный фактор → изменяет условие; измеренный отклик → является наблюдением; статистическая модель → приближает данные; повторные измерения → позволяют оценить разброс. Объясните, почему схема не доказывает причинность автоматически.

**Критерий самопроверки:** независимый коллега, не читавший ваших мыслей, сможет восстановить расчёт и понять, почему вы пока не рекомендовали производство.

In [ ]:
submission={
    "case":"module01_independent_lab",
    "dataset_kind":"synthetic",
    "effects_MPa":effects,
    "converted_units":converted,
    "proposed_experiments":new_experiments,
    "scientific_conclusion":"TODO: напишите здесь обоснованный вывод",
    "limitations":["one observation per condition","no empirical uncertainty estimate"],
}
if effects is None or any(r["strength_MPa"] is None for r in converted) or not new_experiments or submission["scientific_conclusion"].startswith("TODO"):
    print("Отчёт пока не готов. Завершите A1, B1, C1 и текстовые выводы.")
else:
    out=Path("outputs")
    out.mkdir(exist_ok=True)
    target=out/"module01_lab_submission.json"
    target.write_text(json.dumps(submission,ensure_ascii=False,indent=2),encoding="utf-8")
    print("Создан файл для проверки:",target)